# RetailPulse - Interactive Data Cleaning and EDA

This notebook allows for interactive exploration and validation of the raw dataset before it is processed by our automated ETL pipeline (`src/data/make_dataset.py`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Load Raw Data

In [ ]:
# Load the dataset directly from the raw data folder
data_path = '../data/raw/online_retail_II.xlsx'
excel_data = pd.read_excel(data_path, sheet_name=None)
df = pd.concat(excel_data.values(), ignore_index=True)
print(f"Loaded {len(df)} rows.")
df.head()

## 2. Check Missing Values & Anomalies

In [ ]:
print("Missing Values per Column:")
print(df.isnull().sum())

print("\nDescriptive Statistics:")
display(df.describe())

Notice the negative values in `Quantity` and `Price`. These need to be removed as they usually represent returns/cancellations or bad data.

In [ ]:
df_clean = df.dropna(subset=['Customer ID'])
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']
print(f"Rows remaining after cleaning: {len(df_clean)}")

## 3. Basic Visualizations

In [ ]:
# Plot Revenue Distribution (Filtered for extreme outliers)
plt.figure(figsize=(10,6))
sns.histplot(df_clean[df_clean['Revenue'] < 100]['Revenue'], bins=50, kde=True)
plt.title('Distribution of Revenue per Order Line (<$100)')
plt.show()

In [ ]:
# Top 10 Countries by Revenue
top_countries = df_clean.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12,6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='viridis')
plt.title('Top 10 Countries by Total Revenue')
plt.show()